In [1]:
import os
import pandas as pd


import torch
import torch.nn as nn
import torch.optim as optim


import torchvision
import torchvision.transforms as transforms

from torchvision.models import densenet121
from torch.utils.data import DataLoader, random_split


print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cudnn.enabled)

2.2.0+cu121
12.1
True


In [ ]:
#KONIGURaCJa
# dataset:
# "CIFAR10"
# "CIFAR100"
DATASET_NAME = "SVHN"


BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 0.001


MODEL_SAVE_PATH_CHECKPOINT = f"models/t_checkpoint_{DATASET_NAME.lower()}_{BATCH_SIZE}batch_{EPOCHS}epochs_{LEARNING_RATE}lr.pth"
MODEL_SAVE_PATH = f"models/densenet_model_{DATASET_NAME.lower()}_{BATCH_SIZE}batch_{EPOCHS}epochs_{LEARNING_RATE}lr.pth"
RESULTS_TO_EXCEL = "training_results_DENSENET_SVHN.xlsx"
# EARLY STOPPING
PATIENCE = 5

In [3]:
#device
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("======================================")
print("DEVICE:", device)
print("======================================")

DEVICE: cuda


In [4]:
# TRANSFORMACJE
transform_train = transforms.Compose([
    
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4377, 0.4438, 0.4728),
        (0.1980, 0.2010, 0.1970)
    )
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4377, 0.4438, 0.4728),
        (0.1980, 0.2010, 0.1970)
    )
])

In [5]:
# ============================================================
# DATASET
# ============================================================
# ============================================================
# SVHN DATASET
# ============================================================

NUM_CLASSES = 10

full_train_dataset = torchvision.datasets.SVHN(
    root="./data",
    split="train",
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.SVHN(
    root="./data",
    split="test",
    download=True,
    transform=transform_test
)


100%|██████████| 182040794/182040794 [00:37<00:00, 4847332.36it/s]


100%|██████████| 64275384/64275384 [00:38<00:00, 1685207.82it/s]


In [6]:
# ============================================================
# PODZIAŁ TRAIN / VALIDATION
# ============================================================

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

In [7]:
# DATALOADER
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

In [8]:
# MODEL DENSENET121


# weights=None -> brak pretrained weights
model = densenet121(weights=None)

# zmiana pierwszej warstwy dla CIFAR
model.features.conv0 = nn.Conv2d(
    3, #licza kanałów wejściowych (RGB)
    64, # liczba kanałów wyjściowych (zgodna z oryginalnym modelem)
    kernel_size=3,# rozmiar filtra
    stride=1, #krok przesuwania filtra
    padding=1,#dodanie paddingu, żeby zachować rozmiar obrazu bo filtr 3x3 nie miesci sie
    bias=False
)

# usunięcie maxpool
model.features.pool0 = nn.Identity()

# zmiana klasyfikatora
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(  
    model.classifier.in_features,
    NUM_CLASSES)
)

model = model.to(device)

print("======================================")
print(model)
print("======================================")

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): Identity()
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(96, 128, kernel_s

In [9]:

# LOSS + OPTIMIZER


criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)
# optimizer = optim.SGD(
#     model.parameters(),
#     lr=LEARNING_RATE,
#     momentum=0.9,
#     weight_decay=5e-4
# )
# optimizer = optim.Adam(
#     model.parameters(),
#     lr=LEARNING_RATE
# )
# optimizer = optim.SGD(
#     model.parameters(),
#     lr=LEARNING_RATE,
#     momentum=0.9,
#     weight_decay=5e-4
# )
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)
# scheduler = optim.lr_scheduler.StepLR(
#     optimizer,
#     step_size=5,
#     gamma=0.1
# )


scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [10]:

# ============================================================
# EARLY STOPPING VARIABLES
# ============================================================

best_val_loss = float("inf")
early_stop_counter = 0


In [ ]:
#pętla treningowa


print("======================================")
print("START TRENINGU")
print("======================================")

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        # zerowanie gradientów
        optimizer.zero_grad()

        # forward
        outputs = model(images)

        # loss
        loss = criterion(outputs, labels)

        # backward
        loss.backward() #liczenie gradientow

        # update wag
        optimizer.step()

        running_loss += loss.item()
        
        # accuracy train
        _, predicted = torch.max(outputs , 1)

        total_train += labels.size(0)

        correct_train += (predicted == labels).sum().item()
        
        # ====================================================
        # BATCH LOGGING
        # ====================================================

        if (batch_idx + 1) % 100 == 0:

            print(
                f"Epoch [{epoch+1}/{EPOCHS}] | "
                f"Batch [{batch_idx+1}/{len(train_loader)}] | "
                f"Batch Loss: {loss.item():.4f}"
            )

        
    # ========================================================
    # TRAIN METRICS
    # ========================================================

    train_loss = running_loss / len(train_loader)

    train_accuracy = 100 * correct_train / total_train
        
    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    val_loss = running_val_loss / len(val_loader)

    val_accuracy = 100 * correct / total

    # ========================================================
    # SCHEDULER STEP
    # ========================================================

    scheduler.step()
        

    # ========================================================
    # EPOCH LOGGING
    # ========================================================

    current_lr = optimizer.param_groups[0]["lr"]

    print("======================================")
    print(f"Epoch [{epoch+1}/{EPOCHS}] zakończona")
    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Train Accuracy : {train_accuracy:.2f}%")
    print(f"Val Loss       : {val_loss:.4f}")
    print(f"Val Accuracy   : {val_accuracy:.2f}%")
    print(f"Learning Rate  : {current_lr}")
    print("======================================")

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        early_stop_counter = 0

        torch.save(model.state_dict(), MODEL_SAVE_PATH)

        print("Najlepszy model zapisany.")

    else:

        early_stop_counter += 1

        print(f"Brak poprawy. Counter: {early_stop_counter}/{PATIENCE}")

    if early_stop_counter >= PATIENCE:

        print("======================================")
        print("EARLY STOPPING")
        print("======================================")
        break

print("======================================")
print("KONIEC TRENINGU")
print("======================================")

# ============================================================
# WCZYTANIE NAJLEPSZEGO MODELU
# ============================================================

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.to(device)




START TRENINGU
Epoch [1/20] | Batch [100/916] | Batch Loss: 2.0915
Epoch [1/20] | Batch [200/916] | Batch Loss: 1.7380
Epoch [1/20] | Batch [300/916] | Batch Loss: 1.1244
Epoch [1/20] | Batch [400/916] | Batch Loss: 0.8527
Epoch [1/20] | Batch [500/916] | Batch Loss: 0.9138
Epoch [1/20] | Batch [600/916] | Batch Loss: 0.9305
Epoch [1/20] | Batch [700/916] | Batch Loss: 0.8043
Epoch [1/20] | Batch [800/916] | Batch Loss: 0.8769
Epoch [1/20] | Batch [900/916] | Batch Loss: 0.9852
Epoch [1/20] zakończona
Train Loss     : 1.2514
Train Accuracy : 68.09%
Val Loss       : 0.8357
Val Accuracy   : 87.23%
Learning Rate  : 0.0009938441702975688
Najlepszy model zapisany.
Epoch [2/20] | Batch [100/916] | Batch Loss: 0.8448
Epoch [2/20] | Batch [200/916] | Batch Loss: 0.7648
Epoch [2/20] | Batch [300/916] | Batch Loss: 0.7782
Epoch [2/20] | Batch [400/916] | Batch Loss: 0.8728
Epoch [2/20] | Batch [500/916] | Batch Loss: 0.9230
Epoch [2/20] | Batch [600/916] | Batch Loss: 0.7113
Epoch [2/20] | Batch

In [ ]:
# EWALUACJA


model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("======================================")
print(f"Accuracy: {accuracy:.2f}%")
print("======================================")

Accuracy: 64.91%


In [ ]:
# ============================================================
# ZAPIS WYNIKÓW DO EXCELA
# ============================================================

from datetime import datetime

new_result = {
    "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Dataset": DATASET_NAME,
    "Model": "DenseNet121",
    "Scheduler": "CosineAnnealingLR",
    "Optimizer": "AdamW",
    "Batch Size": BATCH_SIZE,
    "Max Epochs": EPOCHS,
    "Executed Epochs": epoch + 1,
    "Learning Rate": LEARNING_RATE,
    "Patience": PATIENCE,
    "Train Accuracy": round(train_accuracy, 2),
    "Validation Accuracy": round(val_accuracy, 2),
    "Test Accuracy": round(accuracy, 2),
    "Best Validation Loss": round(best_val_loss, 4),
    "Model Path": MODEL_SAVE_PATH,
    "Device": str(device)
}

# ============================================================
# ODCZYT ISTNIEJĄCEGO PLIKU
# ============================================================

if os.path.exists(RESULTS_TO_EXCEL):

    try:

        df_existing = pd.read_excel(RESULTS_TO_EXCEL)

        df_new = pd.concat(
            [df_existing, pd.DataFrame([new_result])],
            ignore_index=True
        )

    except Exception as e:

        print("Błąd odczytu Excela:", e)

        print("Tworzenie nowego pliku Excel...")

        df_new = pd.DataFrame([new_result])

else:

    df_new = pd.DataFrame([new_result])

# ============================================================
# ZAPIS EXCEL
# ============================================================

try:

    df_new.to_excel(RESULTS_TO_EXCEL, index=False)

    print("======================================")
    print(f"Wyniki zapisane do: {RESULTS_TO_EXCEL}")
    print("======================================")

except Exception as e:

    print("======================================")
    print("BŁĄD ZAPISU EXCEL")
    print(e)
    print("======================================")

Wyniki zapisane do: training_results.xlsx


In [ ]:
# ============================================================
# ZAPIS MODELU
# ============================================================

checkpoint = {
    "epoch": epoch + 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "best_val_loss": best_val_loss,
    "dataset": DATASET_NAME,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
}

torch.save(checkpoint, MODEL_SAVE_PATH_CHECKPOINT)

print("======================================")
print(f"Model zapisany do: {MODEL_SAVE_PATH_CHECKPOINT}")
print("======================================")

Model zapisany do: models/t_checkpoint_cifar100_32batch_40epochs_0.0001lr.pth
